# Mô hình 1: SARIMAX Baseline

SARIMAX được dùng làm baseline thống kê vì mô hình này mô tả được tự tương quan,
xu hướng và chu kỳ theo thời gian, đồng thời cho phép đưa thêm biến ngoại sinh
như nhiệt độ, độ ẩm, gió và áp suất.

## Bài toán học máy

Mục tiêu là dự báo nồng độ bụi mịn `PM2.5` sau 24 giờ cho Hà Nội, TP.HCM và Đà Nẵng
dựa trên dữ liệu chất lượng không khí, khí tượng, thời gian, đặc trưng trễ và trung bình trượt.

Vì đây là chuỗi thời gian, dữ liệu được chia tuần tự theo `target_time`:
Train 70%, Validation 15%, Test 15%. Validation dùng để theo dõi/tinh chỉnh mô hình;
Test chỉ dùng để đánh giá cuối cùng.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

HORIZON = 24
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Chuẩn bị dữ liệu và chia Train/Validation/Test

In [ ]:
df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values(["city", "datetime"]).reset_index(drop=True)

df["target_pm25_24h"] = df.groupby("city")["pm25"].shift(-HORIZON)
df["target_time"] = df.groupby("city")["datetime"].shift(-HORIZON)

city_name = "Hà Nội"
df_city = df[df["city"] == city_name].dropna(subset=["target_pm25_24h", "target_time"]).copy()
df_city = df_city.sort_values("target_time").reset_index(drop=True)

unique_times = pd.Series(df_city["target_time"].sort_values().unique())
train_cut = unique_times.iloc[int(len(unique_times) * 0.70)]
val_cut = unique_times.iloc[int(len(unique_times) * 0.85)]

train = df_city[df_city["target_time"] < train_cut].copy()
val = df_city[(df_city["target_time"] >= train_cut) & (df_city["target_time"] < val_cut)].copy()
test = df_city[df_city["target_time"] >= val_cut].copy()

split_table = pd.DataFrame(
    {
        "Tập dữ liệu": ["Train", "Validation", "Test"],
        "Số mẫu (giờ)": [len(train), len(val), len(test)],
        "Tỷ lệ (%)": [
            len(train) / len(df_city) * 100,
            len(val) / len(df_city) * 100,
            len(test) / len(df_city) * 100,
        ],
        "Khoảng target_time": [
            f"{train['target_time'].min()} -> {train['target_time'].max()}",
            f"{val['target_time'].min()} -> {val['target_time'].max()}",
            f"{test['target_time'].min()} -> {test['target_time'].max()}",
        ],
    }
)

print("Bảng 1. Tỷ lệ chia dữ liệu cho SARIMAX baseline")
display(split_table)

## Cấu hình huấn luyện

- Target: `target_pm25_24h`.
- Biến ngoại sinh: `pm25`, khí tượng và một số đặc trưng thời gian/trễ.
- Cấu hình SARIMAX: `order=(1,0,1)`, `seasonal_order=(1,0,0,24)`.
- Hàm mất mát được tối ưu gián tiếp qua Maximum Likelihood Estimation; đánh giá bằng RMSE, MAE và R2.

In [ ]:
exog_cols = [
    "pm25", "temp", "humidity", "wind_speed", "pressure",
    "hour", "day_of_week", "month", "pm25_lag_1h", "pm25_lag_24h", "pm25_roll_24h",
]
exog_cols = [col for col in exog_cols if col in df_city.columns]
train = train.dropna(subset=exog_cols + ["target_pm25_24h"]).copy()
val = val.dropna(subset=exog_cols + ["target_pm25_24h"]).copy()
test = test.dropna(subset=exog_cols + ["target_pm25_24h"]).copy()

y_train = train["target_pm25_24h"].astype(float)
y_val = val["target_pm25_24h"].astype(float)
y_test = test["target_pm25_24h"].astype(float)
X_train = train[exog_cols].astype(float)
X_val = val[exog_cols].astype(float)
X_test = test[exog_cols].astype(float)

print("Biến ngoại sinh:", exog_cols)
model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 0, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_result = model.fit(disp=False, maxiter=40)

train_pred = np.asarray(sarimax_result.fittedvalues)
future_exog = pd.concat([X_val, X_test], axis=0)
forecast = sarimax_result.get_forecast(steps=len(future_exog), exog=future_exog).predicted_mean
val_pred = np.asarray(forecast.iloc[: len(y_val)])
test_pred = np.asarray(forecast.iloc[len(y_val) :])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

metrics = pd.DataFrame(
    [
        {
            "model": "SARIMAX",
            "scope": city_name,
            "train_rmse_ug_m3": rmse(y_train, train_pred),
            "val_rmse_ug_m3": rmse(y_val, val_pred),
            "test_rmse_ug_m3": rmse(y_test, test_pred),
            "test_mae_ug_m3": float(mean_absolute_error(y_test, test_pred)),
            "test_r2": float(r2_score(y_test, test_pred)),
        }
    ]
)
metrics.to_csv(RESULTS_DIR / "sarimax_metrics.csv", index=False, encoding="utf-8-sig")
print("Bảng 2. Chỉ số đánh giá SARIMAX, đơn vị lỗi: µg/m³")
display(metrics)

In [ ]:
plot_df = test[["target_time"]].copy()
plot_df["actual_pm25"] = y_test.to_numpy()
plot_df["predicted_pm25"] = test_pred
plot_df = plot_df.head(240)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_df["target_time"], plot_df["actual_pm25"], label="Thực tế")
ax.plot(plot_df["target_time"], plot_df["predicted_pm25"], label="Dự đoán SARIMAX")
ax.set_title("SARIMAX - Dự báo PM2.5 trên 240 giờ đầu của tập Test")
ax.set_xlabel("Target time")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
fig_path = FIGURES_DIR / "sarimax_test_forecast.png"
plt.savefig(fig_path, dpi=180)
plt.show()

Hình 1. Đường dự báo SARIMAX so với PM2.5 thực tế trên tập Test.

SARIMAX không có learning curve theo epoch vì tham số được ước lượng bằng Maximum Likelihood một lần.
Notebook XGBoost và LSTM bên dưới cung cấp learning curve theo boosting round/epoch.